# Week 4: Query Parser to SQL

Goal: turn plain-English searches like "3 bed under 700k in Irvine" into filters, check them against the real data, and build a safe SQL query.

1. Example queries
2. Accuracy on the Week 1 sample queries
3. SQL generation
4. Schema validation
5. SQL injection check

In [1]:
import json
import sqlite3
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from scripts.query_parser import QueryParser, SchemaValidator

parser = QueryParser()
validator = SchemaValidator()
pd.set_option("display.max_colwidth", 120)

## 1. Example queries

One example per pattern the parser handles.

In [2]:
examples = [
    "3 bed under 700k in Irvine",
    "3+ bedrooms, at least 2.5 baths in Pasadena",
    "2-3 bedroom condo between $500k and $800k",
    "around $900k in San Diego",
    "over 2000 sq ft built after 2010",
    "townhouse with a 2-car garage and HOA under $300",
    "single-story house in Irvine with no HOA",
    "home with an ADU or guest house in LA",
    "not a condo, without a pool",
    "mid-century modern with mountain views in Palm Springs",
]
pd.DataFrame({"query": examples, "filters": [parser.parse(q) for q in examples]})

,query,filters
0,3 bed under 700k in Irvine,"{'bedrooms': 3, 'price_max': 700000, 'city': ['Irvine']}"
1,"3+ bedrooms, at least 2.5 baths in Pasadena","{'bedrooms_min': 3, 'bathrooms_min': 2.5, 'city': ['Pasadena']}"
2,2-3 bedroom condo between $500k and $800k,"{'bedrooms_min': 2, 'bedrooms_max': 3, 'price_min': 500000, 'price_max': 800000, 'property_type': ['Condominium']}"
3,around $900k in San Diego,"{'price_target': 900000, 'city': ['San Diego']}"
4,over 2000 sq ft built after 2010,"{'year_built_min': 2011, 'sqft_min': 2000}"
5,townhouse with a 2-car garage and HOA under $300,"{'hoa_max': 300, 'garage_spaces_min': 2, 'property_type': ['Townhouse']}"
6,single-story house in Irvine with no HOA,"{'property_type': ['SingleFamilyResidence'], 'city': ['Irvine'], 'amenities': ['single-story'], 'exclude_amenities':..."
7,home with an ADU or guest house in LA,"{'city': ['Los Angeles'], 'amenities_any': ['adu', 'guest house']}"
8,"not a condo, without a pool","{'exclude_property_type': ['Condominium'], 'exclude_amenities': ['pool']}"
9,mid-century modern with mountain views in Palm Springs,"{'city': ['Palm Springs'], 'amenities': ['mid-century modern', 'mountain view']}"


## 2. Accuracy on the Week 1 sample queries

I wrote down the correct filters for all 62 queries before building the parser (`data/processed/sample_queries_parsed.json`). A query only counts as correct if every filter matches exactly.

**Conventions**
- "this condo" or "these townhomes" point at a specific listing, so they aren't search filters
- Anything from the Week 1 taxonomy counts as an amenity, even in questions ("is the backyard fenced")

In [3]:
queries = pd.read_csv(ROOT / "data/processed/sample_queries.csv")
gold = json.loads((ROOT / "data/processed/sample_queries_parsed.json").read_text())

def same(a, b):
    norm = lambda d: {k: sorted(v) if isinstance(v, list) else v for k, v in d.items()}
    return norm(a) == norm(b)

queries["parsed"] = queries["query"].apply(parser.parse)
queries["expected"] = queries["query_id"].map(gold)
queries["correct"] = [same(p, e) for p, e in zip(queries["parsed"], queries["expected"])]
print(f"Accuracy: {queries['correct'].mean():.1%} ({queries['correct'].sum()} of {len(queries)})")
queries.groupby("intent")["correct"].agg(["sum", "count"]).rename(columns={"sum": "correct", "count": "queries"})

Accuracy: 98.4% (61 of 62)


,correct,queries
intent,,
compare_listings,7,7
financing_costs,8,8
neighborhood_info,7,8
pricing_valuation,8,8
property_details,9,9
schedule_showing,7,7
search_listings,15,15


In [4]:
# The one miss
queries[~queries["correct"]][["query", "parsed", "expected"]]

,query,parsed,expected
44,how close is the nearest park,{},{'amenities': ['parks']}


The taxonomy has "parks" but the query says "park". I left this as a miss instead of tuning the parser to the test.

In [5]:
# All search queries and what they parse to
queries[queries["intent"] == "search_listings"][["query", "parsed"]]

,query,parsed
0,3 bedroom house with a pool in San Diego under $1.2M,"{'bedrooms': 3, 'price_max': 1200000, 'property_type': ['SingleFamilyResidence'], 'city': ['San Diego'], 'amenities'..."
1,condos near the beach in Santa Monica with ocean views,"{'property_type': ['Condominium'], 'city': ['Santa Monica'], 'amenities': ['near the beach', 'ocean view']}"
2,single-story homes in Irvine with no HOA,"{'city': ['Irvine'], 'amenities': ['single-story'], 'exclude_amenities': ['hoa']}"
3,townhouse with an attached 2-car garage in San Jose,"{'garage_spaces_min': 2, 'property_type': ['Townhouse'], 'city': ['San Jose']}"
4,move-in ready homes in Pasadena with a large backyard,"{'city': ['Pasadena'], 'amenities': ['move-in ready', 'backyard']}"
5,homes with an ADU or guest house in Los Angeles,"{'city': ['Los Angeles'], 'amenities_any': ['adu', 'guest house']}"
6,fixer-upper under 700k in Sacramento,"{'price_max': 700000, 'city': ['Sacramento'], 'amenities': ['fixer-upper']}"
7,4 bed 3 bath with a home office near good schools in Fremont,"{'bedrooms': 4, 'bathrooms': 3, 'city': ['Fremont'], 'amenities': ['home office', 'schools']}"
8,pet-friendly condo with in-unit laundry in downtown LA,"{'property_type': ['Condominium'], 'city': ['Los Angeles'], 'amenities': ['pet-friendly', 'laundry room', 'downtown']}"
9,homes with solar panels and EV charging in Riverside,"{'city': ['Riverside'], 'amenities': ['solar panels', 'ev charger']}"


## 3. SQL generation

Column names come from a fixed list and every value goes into `params`. The database fills them in separately, so nothing a user types becomes part of the SQL.

In [6]:
filters = parser.parse("3 bed condo under 700k in Irvine with a pool, no HOA")
sql, params = parser.to_sql(filters)
print("Filters:", filters)
print("\nSQL:   ", sql)
print("Params:", params)

Filters: {'bedrooms': 3, 'price_max': 700000, 'property_type': ['Condominium'], 'city': ['Irvine'], 'amenities': ['pool'], 'exclude_amenities': ['hoa']}

SQL:    SELECT L_ListingID, L_Address, L_City, L_Type_, L_SystemPrice, L_Keyword2, LM_Dec_3, LM_Int2_3 FROM rets_property WHERE L_SystemPrice <= %s AND L_Keyword2 = %s AND L_City IN (%s) AND L_Type_ IN (%s) AND PoolPrivateYN = %s AND (AssociationYN IS NULL OR AssociationYN <> %s) LIMIT %s
Params: [700000, 3, 'Irvine', 'Condominium', '1', 1, 50]


Amenities with their own database column (pool, spa, fireplace, HOA, stories, ...) use that column. Everything else searches the listing description for the term and its synonyms:

In [7]:
sql, params = parser.to_sql(parser.parse("home with solar panels"))
print(sql.split("WHERE")[1])
print(params)

 (L_Remarks LIKE %s OR L_Remarks LIKE %s OR L_Remarks LIKE %s OR L_Remarks LIKE %s) LIMIT %s
['%solar panels%', '%solar%', '%owned solar%', '%leased solar%', 50]


## 4. Schema validation

The validator checks filters against `data/schema.json` (real cities and property types from the database, plus sensible ranges) before any SQL is run.

In [8]:
tests = [
    "3 bed in Portland under 500k",
    "townhome in irvin under 500k",
    "25 bedroom house",
    "condo under 50k",
    "between $900k and $500k",
    "3 bed under 700k in Irvine",
]
rows = []
for q in tests:
    ok, errors = validator.validate_query(parser.parse(q))
    rows.append({"query": q, "valid": ok, "message": " ".join(errors) or "OK"})
pd.DataFrame(rows)

,query,valid,message
0,3 bed in Portland under 500k,False,City 'Portland' not found in the database.
1,townhome in irvin under 500k,False,City 'irvin' not found in the database. Did you mean Irvine?
2,25 bedroom house,False,Bedrooms 25 is outside the expected range (0 to 10)
3,condo under 50k,False,"Price max 50,000 is outside the expected range (100,000 to 10,000,000)"
4,between $900k and $500k,False,price_min is greater than price_max
5,3 bed under 700k in Irvine,True,OK


## 5. SQL injection check

I ran the generated SQL against a small in-memory database with attack strings as values. They're treated as plain text, so they match nothing and the table stays intact.

In [9]:
db = sqlite3.connect(":memory:")
db.execute("CREATE TABLE rets_property (L_ListingID, L_Address, L_City, L_Type_, L_SystemPrice, L_Keyword2, LM_Dec_3, LM_Int2_3)")
db.executemany("INSERT INTO rets_property VALUES (?,?,?,?,?,?,?,?)", [
    ("1", "1 A St", "Irvine", "Condominium", 650000, 3, 2, 1400),
    ("2", "2 B St", "Irvine", "SingleFamilyResidence", 1500000, 4, 3, 2600),
])

attacks = [
    "Irvine'; DROP TABLE rets_property; --",
    "Irvine' OR '1'='1",
    "Irvine'); DELETE FROM rets_property; --",
]
rows = []
for attack in attacks:
    sql, params = parser.to_sql({"city": [attack]})   # skip validation on purpose
    matches = db.execute(sql.replace("%s", "?"), params).fetchall()
    remaining = db.execute("SELECT COUNT(*) FROM rets_property").fetchone()[0]
    rows.append({"attack": attack, "in SQL text?": attack in sql, "rows matched": len(matches),
                 "rows left in table": remaining})
pd.DataFrame(rows)

,attack,in SQL text?,rows matched,rows left in table
0,Irvine'; DROP TABLE rets_property; --,False,0,2
1,Irvine' OR '1'='1,False,0,2
2,Irvine'); DELETE FROM rets_property; --,False,0,2


In [10]:
# Made-up filter names can't sneak in a column either
try:
    parser.to_sql({"L_City = 'x' OR 1=1 --": 1})
except ValueError as e:
    print("Blocked:", e)

Blocked: Unknown filter(s): ["L_City = 'x' OR 1=1 --"]


## 6. Key findings

**I. Results**

*The parser beats the 90% target on the Week 1 queries.*
- 98.4% accuracy (61 of 62 parsed exactly right)
- 20+ patterns covered, including price ranges, bed/bath min and max, amenities, and negation
- 97 test cases in `tests/test_week4.py`

**II. Safety**

*User text never becomes part of the SQL.*
- Every value is passed as a parameter, and column names come from a fixed list
- Attack strings matched nothing and the table stayed intact
- The validator catches unknown cities (with a "Did you mean...?" hint), out-of-range prices and bedrooms, and min above max

**III. What was tricky**

*Real queries mix filters with everyday words.*
- City names overlap amenities ("Palm Desert" is not a "desert" amenity, "Mountain View" is not a view)
- Some city names are common words (Orange, Vista, Paradise), so they only count after "in" or "near"
- "open house" and "guest house" aren't houses, and "this condo" means a specific listing, not a search

**IV. Next steps**
- Add singular forms like "park" to the taxonomy
- Week 7's intent classifier can decide when amenities in a question ("is the backyard fenced") should become search filters